In [1]:
def preprocess_sentence_kr(w):
  w = w.strip()
  w = re.sub(r"[^가-힣?.!,¿]+", " ", w) 
  w = w.strip() 
  return w

In [2]:
import urllib.request
import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote
from tqdm import tqdm
import time

def naver_blog_crawling_txt_split(keyword, txt_name, total_count=1000):
    print(f"[{keyword}] 마스터의 명에 따라 데이터를 '||'로 분할하여 '{txt_name}'에 저장합니다!\n")
    
    # 💡 마스터님의 API 키
    client_id = "S4Znt5f9gb1guLPgjT69" 
    client_secret = "Erk25kIud0"
    
    encText = quote(keyword)
    href_list = []
    
    # ==========================================
    # 1. API 반복 호출 (URL 수집)
    # ==========================================
    max_start = min(total_count, 1000) 
    print(">>> 1단계: API를 통해 모바일 링크를 긁어모으고 있습니다...")
    
    for start_idx in range(1, max_start + 1, 100):
        display_count = min(100, max_start - start_idx + 1)
        url = f"https://openapi.naver.com/v1/search/blog?query={encText}&display={display_count}&start={start_idx}&sort=sim"
        
        request = urllib.request.Request(url)
        request.add_header("X-Naver-Client-Id", client_id)
        request.add_header("X-Naver-Client-Secret", client_secret)
        
        try:
            response = urllib.request.urlopen(request)
            if response.getcode() == 200:
                data = json.loads(response.read().decode('utf-8'))
                for item in data['items']:
                    original_link = item['link']
                    mobile_link = original_link.replace("blog.naver.com", "m.blog.naver.com")
                    href_list.append(mobile_link)
            else:
                break
        except:
            break
        time.sleep(0.1) 
        
    print(f"✔️ 총 {len(href_list)}개의 링크 확보 완료!\n")

    # ==========================================
    # 2. 본문 추출 및 '||' 구분자로 txt 파일에 바로 쓰기
    # ==========================================
    print(f">>> 2단계: 본문을 추출하여 메모장에 기록합니다...")
    headers = {"User-Agent": "Mozilla/5.0"}
    
    # 💡 파일을 쓰기 모드('w')로 열어둡니다.
    with open(txt_name, 'w', encoding='utf-8') as f:
        f.write('크롤링워드||url||content\n')
        for href in tqdm(href_list, desc="본문 추출 진행률"):
            try:
                res = requests.get(href, headers=headers)
                soup = BeautifulSoup(res.text, 'html.parser')
                
                content_element = soup.select_one('.se-main-container')
                if not content_element:
                    content_element = soup.select_one('#postViewArea')
                
                if content_element:
                    # 본문을 가져옵니다.
                    content = content_element.get_text(separator=' ', strip=True)
                    
                    # 💡 장인의 방어막: 본문 안의 줄바꿈(\n, \r)을 모두 공백으로 없애버립니다.
                    # 그래야 텍스트 파일에서 딱 한 줄을 차지하게 되어 나중에 분할하기 좋습니다.
                    clean_content = content.replace('\n', ' ').replace('\r', '')
                    
                    # 💡 [키워드 || URL || 본문내용] 순서로 쓴 뒤 줄바꿈(\n)을 해줍니다.
                    line_to_write = f"{keyword}||{href}||{clean_content}\n"
                    f.write(line_to_write)
                    
                time.sleep(0.2) 
            except:
                continue
                
    print(f"\n[임무 완수] 모든 데이터가 '{txt_name}' 파일에 완벽하게 분할 기록되었습니다, 마스터!")

# ==========================================
# 🚀 함수 실행 방법
# ==========================================
# naver_blog_crawling_txt_split('육아용품 기간', 'naver_blog_split_data.txt', 1000)

In [3]:
naver_blog_crawling_txt_split('육아제품구독', './data/naver_blog_contents_육아제품구독.txt', 10000)

[육아제품구독] 마스터의 명에 따라 데이터를 '||'로 분할하여 './data/naver_blog_contents_육아제품구독.txt'에 저장합니다!

>>> 1단계: API를 통해 모바일 링크를 긁어모으고 있습니다...
✔️ 총 1000개의 링크 확보 완료!

>>> 2단계: 본문을 추출하여 메모장에 기록합니다...


본문 추출 진행률: 100%|██████████| 1000/1000 [20:15<00:00,  1.22s/it]


[임무 완수] 모든 데이터가 './data/naver_blog_contents_육아제품구독.txt' 파일에 완벽하게 분할 기록되었습니다, 마스터!


In [3]:
naver_blog_crawling_txt_split('생애 주기별 육아제품', './data/naver_blog_contents_생애주기별육아제품.txt', 10000)

[생애 주기별 육아제품] 마스터의 명에 따라 데이터를 '||'로 분할하여 './data/naver_blog_contents_생애주기별육아제품.txt'에 저장합니다!

>>> 1단계: API를 통해 모바일 링크를 긁어모으고 있습니다...
✔️ 총 944개의 링크 확보 완료!

>>> 2단계: 본문을 추출하여 메모장에 기록합니다...


본문 추출 진행률: 100%|██████████| 944/944 [21:16<00:00,  1.35s/it]


[임무 완수] 모든 데이터가 './data/naver_blog_contents_생애주기별육아제품.txt' 파일에 완벽하게 분할 기록되었습니다, 마스터!
